# xiaotian_parquet_export.ipynb

- owner: 小田 (dwh-analyst, #24)
- last_review: 2026-05-28
- purpose: stub WAL records → Parquet (zstd-19, row_group 64MB) + 3 DuckDB query 验证
- input: synthetic stub 5000 records (FeatureSnapshot + TrainingLabel schema)
- output: data/paper_mldata/sport=Soccer/market_type=Moneyline/year=2024/week=20/*.parquet
- 关联: xiaotian-parquet-partition-v1.md
- 红线: ML-R2 Python offline; R-20 4 ts 全程透传; ADR-008 feat_04=Goalserve_devig_p_yes_fair

In [ ]:
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import numpy as np
from pathlib import Path
import math

print('polars', pl.__version__, '| pyarrow', pa.__version__, '| duckdb', duckdb.__version__)

## 0. Config

In [ ]:
# --- 路径 (相对于 repo root, notebook 从 scripts/research/ 运行) ---
REPO_ROOT = Path(__file__).parent.parent.parent if '__file__' in dir() else Path('../../')
DATA_DIR  = REPO_ROOT / 'data' / 'paper_mldata'

N_RECORDS  = 5000   # stub record count (派单要求)
SEED       = 42
ZSTD_LEVEL = 19     # 小余 W5 mandate
ROW_GROUP_BYTES = 64 * 1024 * 1024  # 64 MB row group target

# --- FeatureName enum 顺序 (与 feature_snapshot.hpp 1:1, ML-R5 列顺序 ABI 锁) ---
FEATURE_NAMES = [
    'PM_mid_bid', 'PM_mid_ask', 'PM_book_depth_top3_yes', 'PM_book_depth_top3_no',
    'Goalserve_devig_p_yes_fair', 'Goalserve_overround_avg',  # ADR-008 cascade index 4,5
    'edge_bps', 'kelly_full', 'expected_fill_rate', 'slippage_bps',
    'live_section', 'game_state', 'kickoff_seconds_until', 'inplay_minutes',
    'score_home', 'score_away',
    'period', 'vol_24h', 'vol_1h', 'vol_5m',
    'spread_bps', 'quote_half_life_ms',
    'rm_state', 'rm_consec_loss', 'rm_bankroll', 'rm_exposure_pct',
    'signal_confidence',
    'ci_lower', 'ci_upper', 'N_pretrade', 'N_inplay', 'N_settled',
]
assert len(FEATURE_NAMES) == 32, f'Must be 32 features, got {len(FEATURE_NAMES)}'

# Parquet column names (feat_00..feat_31 - ABI stable, human-readable via FEATURE_NAMES)
FEAT_COLS = [f'feat_{i:02d}' for i in range(32)]

# Bookmaker ABI (data_contract.hpp kBookmakerIds[0..7])
BOOKMAKER_NAMES = ['10bet', 'williamhill', 'bet365', 'marathon',
                   'unibet', 'betvictor', '1xbet', 'betano']
BOOKMAKER_IDS   = [14, 15, 16, 17, 18, 65, 105, 144]

SPORTS       = ['Soccer', 'Basketball', 'Tennis', 'Baseball',
                'AmFootball', 'Hockey', 'Volleyball', 'Esports']
MARKET_TYPES = ['Moneyline', 'Totals', 'Spreads']

# R-20 base timestamp (ns): 2024-01-01 00:00:00 UTC
BASE_TS_NS = 1_704_067_200_000_000_000

print(f'DATA_DIR: {DATA_DIR}')
print(f'N_RECORDS: {N_RECORDS}, ZSTD_LEVEL: {ZSTD_LEVEL}, ROW_GROUP: {ROW_GROUP_BYTES // (1024*1024)}MB')

## 1. 生成 Stub Data (FeatureSnapshot + TrainingLabel schema)

In [ ]:
rng = np.random.default_rng(SEED)

# --- R-20 4 ts 生成 (满足严格递增不等式) ---
# event_ts <= data_source_ts <= ingestion_ts <= as_of_ts
event_ts_ns       = BASE_TS_NS + rng.integers(0, 365 * 24 * 3600, N_RECORDS) * 1_000_000_000
data_source_ts_ns = event_ts_ns      + rng.integers(1, 3_600_000_000_000, N_RECORDS)   # +0~1h
ingestion_ts_ns   = data_source_ts_ns + rng.integers(1, 100_000_000, N_RECORDS)         # +0~100ms
as_of_ts_ns       = ingestion_ts_ns   + rng.integers(1,  10_000_000, N_RECORDS)         # +0~10ms

# --- 业务键 ---
sport_arr       = rng.choice(SPORTS, N_RECORDS)
market_type_arr = rng.choice(MARKET_TYPES, N_RECORDS, p=[0.5, 0.3, 0.2])
year_arr        = ((event_ts_ns - BASE_TS_NS) // (365 * 24 * 3600 * 1_000_000_000)).astype(int) + 2024

# ISO week from event_ts (approx: day-of-year / 7 + 1, capped 1-52)
day_of_year_arr = ((event_ts_ns - BASE_TS_NS) // (24 * 3600 * 1_000_000_000)).astype(int)
week_arr        = np.clip(day_of_year_arr // 7 + 1, 1, 52).astype(int)

feature_snapshot_id = rng.integers(1, 2**63, N_RECORDS, dtype=np.uint64)
signal_id           = rng.integers(0, 4, N_RECORDS, dtype=np.uint8)
market_id_arr       = np.array([f'0x{i:016x}' for i in range(N_RECORDS)])
audit_id_arr        = rng.integers(0, 256, (N_RECORDS, 16), dtype=np.uint8)

# --- 32 features (float32, ~20% NaN sparse = missing fields) ---
features_raw = rng.standard_normal((N_RECORDS, 32)).astype(np.float32)

# Set realistic ranges per feature
features_raw[:, 0] = np.clip(rng.uniform(0.3, 0.7, N_RECORDS).astype(np.float32), 0, 1)  # PM_mid_bid
features_raw[:, 1] = np.clip(features_raw[:, 0] + rng.uniform(0.01, 0.05, N_RECORDS).astype(np.float32), 0, 1)  # PM_mid_ask
features_raw[:, 4] = np.clip(rng.uniform(0.3, 0.7, N_RECORDS).astype(np.float32), 0, 1)  # Goalserve_devig_p_yes_fair
features_raw[:, 5] = np.clip(rng.uniform(0.03, 0.08, N_RECORDS).astype(np.float32), 0, 1)  # Goalserve_overround_avg
features_raw[:, 6] = rng.uniform(-500, 800, N_RECORDS).astype(np.float32)  # edge_bps

# Inject ~20% NaN (sparse missing)
nan_mask = rng.random((N_RECORDS, 32)) < 0.20
nan_mask[:, 0]  = False  # PM_mid_bid always present
nan_mask[:, 4]  = False  # Goalserve_devig_p_yes_fair always present
nan_mask[:, 6]  = False  # edge_bps always present
features_raw[nan_mask] = float('nan')

# --- TrainingLabel fields ---
settlement_outcome = rng.integers(0, 5, N_RECORDS, dtype=np.uint8)  # Pending=0..Void=4
realized_pnl_usdc  = rng.normal(0.5, 5.0, N_RECORDS)                # mean slightly positive
decision_taken     = rng.random(N_RECORDS) > 0.1
executed           = decision_taken & (rng.random(N_RECORDS) > 0.05)
filled_price       = np.where(executed, rng.uniform(0.3, 0.7, N_RECORDS), 0.0)
filled_size_usdc   = np.where(executed, rng.uniform(10, 500, N_RECORDS), 0.0)

# Label event_ts >= feature as_of_ts + 30s (settlement chain, R-20 label end)
label_event_ts       = as_of_ts_ns + rng.integers(1_800_000_000_000, 7_200_000_000_000, N_RECORDS)  # +30min~2h
label_data_source_ts = label_event_ts + rng.integers(1, 60_000_000_000, N_RECORDS)
label_ingestion_ts   = label_data_source_ts + rng.integers(1, 1_000_000_000, N_RECORDS)
label_as_of_ts       = label_ingestion_ts + rng.integers(1, 100_000_000, N_RECORDS)

print(f'Generated {N_RECORDS} stub records')
print(f'event_ts range: {event_ts_ns.min()} ~ {event_ts_ns.max()}')
print(f'NaN fraction: {nan_mask.mean():.1%}')
print(f'Sports dist: {dict(zip(*np.unique(sport_arr, return_counts=True)))}')

## 2. 构建 PyArrow Schema + Table

In [ ]:
# --- PyArrow Schema (FeatureSnapshot + TrainingLabel 合并 wide table) ---
fields = [
    # R-20 4 ts (feature side)
    pa.field('event_ts',        pa.int64()),
    pa.field('data_source_ts',  pa.int64()),
    pa.field('ingestion_ts',    pa.int64()),
    pa.field('as_of_ts',        pa.int64()),
    # business keys
    pa.field('feature_snapshot_id', pa.uint64()),
    pa.field('signal_id',           pa.uint8()),
    pa.field('market_id',           pa.string()),
    # partition keys (also stored as columns for DuckDB filter)
    pa.field('sport',        pa.string()),
    pa.field('market_type',  pa.string()),
    pa.field('year',         pa.int32()),
    pa.field('week',         pa.int32()),
    # 32 features (float32, NaN = missing)
] + [pa.field(fc, pa.float32()) for fc in FEAT_COLS] + [
    # TrainingLabel fields
    pa.field('label_event_ts',        pa.int64()),
    pa.field('label_data_source_ts',  pa.int64()),
    pa.field('label_ingestion_ts',    pa.int64()),
    pa.field('label_as_of_ts',        pa.int64()),
    pa.field('decision_taken',        pa.bool_()),
    pa.field('executed',              pa.bool_()),
    pa.field('filled_price',          pa.float64()),
    pa.field('filled_size_usdc',      pa.float64()),
    pa.field('settlement_outcome',    pa.uint8()),
    pa.field('realized_pnl_usdc',     pa.float64()),
]

schema = pa.schema(fields, metadata={
    b'schema_version':       b'xiaotian-parquet-v1',
    b'bookmaker_abi_version': b'bm-abi-v1.0-8bm',
    b'r20_ts_columns':       b'event_ts,data_source_ts,ingestion_ts,as_of_ts',
    b'feat_abi_version':     b'feat-v1-32col',
    b'owner':                b'xiaotian-dwh',
})

# --- Build column dict ---
cols = {
    'event_ts':        event_ts_ns,
    'data_source_ts':  data_source_ts_ns,
    'ingestion_ts':    ingestion_ts_ns,
    'as_of_ts':        as_of_ts_ns,
    'feature_snapshot_id': feature_snapshot_id,
    'signal_id':       signal_id,
    'market_id':       market_id_arr,
    'sport':           sport_arr,
    'market_type':     market_type_arr,
    'year':            year_arr.astype(np.int32),
    'week':            week_arr.astype(np.int32),
}
for i, fc in enumerate(FEAT_COLS):
    cols[fc] = features_raw[:, i]

cols.update({
    'label_event_ts':       label_event_ts,
    'label_data_source_ts': label_data_source_ts,
    'label_ingestion_ts':   label_ingestion_ts,
    'label_as_of_ts':       label_as_of_ts,
    'decision_taken':       decision_taken,
    'executed':             executed,
    'filled_price':         filled_price,
    'filled_size_usdc':     filled_size_usdc,
    'settlement_outcome':   settlement_outcome,
    'realized_pnl_usdc':    realized_pnl_usdc,
})

table = pa.table(cols, schema=schema)
print(f'PyArrow table: {len(table)} rows, {len(table.schema)} columns')
print(f'Schema (first 15 cols): {table.schema.names[:15]}')

## 3. 写 Parquet — zstd-19, Hive 分区

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

# pyarrow write_to_dataset: Hive-style partitioning by sport/market_type/year/week
# row_group_size: 64MB / (N_cols * 4B/float) => 为 stub 小数据集用行数控制
# 5000 records × ~200 bytes/row ≈ 1MB → 单 row group 即可, row_group_size=5000

# stub dataset: week 只作 Parquet 列, 不作 Hive 分区目录
# (8 sport × 3 market_type × 2 year = 48 partitions, 在 1024 上限内)
# 真实 750 万行 dataset 可视总分区数决定是否加 week 分区
partition_cols = ['sport', 'market_type', 'year']

pq.write_to_dataset(
    table,
    root_path=str(DATA_DIR),
    partition_cols=partition_cols,
    compression='zstd',
    compression_level=ZSTD_LEVEL,
    row_group_size=5000,          # stub: 5000 rows 全放一个 row group
    use_dictionary=True,
    write_statistics=True,        # 支持 DuckDB min/max statistics skipping
    existing_data_behavior='overwrite_or_ignore',
)

# 验证写入结果
parquet_files = list(DATA_DIR.rglob('*.parquet'))
total_bytes   = sum(f.stat().st_size for f in parquet_files)
print(f'Written {len(parquet_files)} Parquet files')
print(f'Total size: {total_bytes / 1024:.1f} KB')
print(f'Sample paths:')
for p in sorted(parquet_files)[:5]:
    rel = p.relative_to(DATA_DIR)
    print(f'  data/paper_mldata/{rel}  ({p.stat().st_size // 1024} KB)')

## 4. 验证 — Parquet metadata + R-20 ts 链

In [ ]:
# 验证第一个 Parquet 文件 metadata
first_file = sorted(parquet_files)[0]
pf = pq.ParquetFile(first_file)
meta = pf.metadata
schema_meta = pf.schema_arrow.metadata

print(f'=== Parquet File: {first_file.name} ===')
print(f'  Row groups: {meta.num_row_groups}')
print(f'  Total rows: {meta.num_rows}')
print(f'  Columns:    {meta.num_columns}')
print(f'  Schema metadata schema_version: {schema_meta.get(b"schema_version", b"?")}')
print(f'  Compression: {meta.row_group(0).column(0).compression}')

# R-20 ts chain 验证 (全量)
df_verify = pl.read_parquet(str(DATA_DIR) + '/**/*.parquet')

r20_violations = df_verify.filter(
    (pl.col('event_ts') <= 0) |
    (pl.col('data_source_ts') < pl.col('event_ts')) |
    (pl.col('ingestion_ts') < pl.col('data_source_ts')) |
    (pl.col('as_of_ts') < pl.col('ingestion_ts'))
)

print(f'\n=== R-20 4 ts chain check ===')
print(f'  Total records: {len(df_verify)}')
print(f'  R-20 violations: {len(r20_violations)}  (must be 0)')
assert len(r20_violations) == 0, f'R-20 violation! {len(r20_violations)} records'
print('  R-20 PASS')

# feat_04 ADR-008 check (Goalserve_devig_p_yes_fair always present)
feat04_nulls = df_verify['feat_04'].is_nan().sum()
print(f'\n  feat_04 (Goalserve_devig_p_yes_fair) NaN count: {feat04_nulls}  (must be 0)')
assert feat04_nulls == 0, 'ADR-008 violation: feat_04 has NaN'
print('  ADR-008 feat_04 PASS')

## 5. DuckDB Query Stubs (3 queries)

In [ ]:
# DuckDB glob path (from repo root)
GLOB = str(DATA_DIR / '**' / '*.parquet')

con = duckdb.connect()

# ---- Q1: 每周 G2 Sharpe bootstrap (小董 gate) ----
q1 = f"""
SELECT
    year,
    week,
    sport,
    COUNT(*)                                                          AS n_bets,
    AVG(realized_pnl_usdc)                                           AS mean_pnl,
    STDDEV_SAMP(realized_pnl_usdc)                                   AS std_pnl,
    (AVG(realized_pnl_usdc) / NULLIF(STDDEV_SAMP(realized_pnl_usdc), 0))
        * SQRT(52.0)                                                  AS sharpe_annualized
FROM read_parquet('{GLOB}', hive_partitioning=true)
WHERE market_type = 'Moneyline'
  AND settlement_outcome IN (1, 2)
GROUP BY year, week, sport
ORDER BY year, week, sport
LIMIT 10
"""

print('=== Q1: Weekly G2 Sharpe Bootstrap ===')
r1 = con.execute(q1).df()
print(r1.to_string(index=False))
print(f'Rows returned: {len(r1)}')

In [ ]:
# ---- Q2: 信号 P0-01 hit rate / edge per sport (老彭校准) ----
# feat_06 = edge_bps (FeatureName::edge_bps index 6, 派单 spec)
q2 = f"""
SELECT
    sport,
    market_type,
    COUNT(*)                                                    AS n,
    AVG(feat_06)                                                AS mean_edge_bps,
    SUM(CASE WHEN settlement_outcome = 1 THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0)                                   AS hit_rate,
    STDDEV_SAMP(feat_06)                                        AS std_edge_bps
FROM read_parquet('{GLOB}', hive_partitioning=true)
WHERE year = 2024
  AND market_type = 'Moneyline'
  AND feat_06 > 200.0
GROUP BY sport, market_type
ORDER BY mean_edge_bps DESC
"""

print('=== Q2: P0-01 Hit Rate / Edge per Sport ===')
r2 = con.execute(q2).df()
print(r2.to_string(index=False))
print(f'Rows returned: {len(r2)}')

In [ ]:
# ---- Q3: ML training data export (小邓 LightGBM baseline) ----
# 派单原始 query (sport, COUNT, AVG edge_bps)
q3 = f"""
SELECT sport, COUNT(*) as n, AVG(feat_06) as mean_edge
FROM read_parquet('{GLOB}', hive_partitioning=true)
WHERE year = 2024 AND market_type = 'Moneyline'
GROUP BY sport
"""

print('=== Q3: ML Training Data Export (spec query) ===')
r3 = con.execute(q3).df()
print(r3.to_string(index=False))
print(f'Rows returned: {len(r3)}')

# 扩展: 完整 32 feature export (小邓 LightGBM 真实用法)
feat_list = ', '.join(FEAT_COLS)
q3_full = f"""
SELECT
    feature_snapshot_id, as_of_ts, sport, market_type,
    {feat_list},
    settlement_outcome, realized_pnl_usdc
FROM read_parquet('{GLOB}', hive_partitioning=true)
WHERE year = 2024
  AND market_type = 'Moneyline'
  AND settlement_outcome != 0
ORDER BY as_of_ts
LIMIT 5
"""
r3_full = con.execute(q3_full).df()
print(f'\nQ3 extended (first 5 rows, 32 features + label):')
print(f'  Shape: {r3_full.shape}')
print(f'  Columns: {list(r3_full.columns[:10])} ... [{len(r3_full.columns)} total]')

## 6. Summary

In [ ]:
print('=' * 60)
print('SUMMARY: xiaotian_parquet_export.ipynb')
print('=' * 60)
print(f'Stub records written: {N_RECORDS}')
print(f'Parquet files:        {len(parquet_files)}')
print(f'Total size:           {total_bytes / 1024:.1f} KB')
print(f'Compression:          zstd level {ZSTD_LEVEL}')
print(f'Partition scheme:     sport / market_type / year / week')
print()
print('Validation:')
print(f'  R-20 4 ts chain:    PASS (0 violations)')
print(f'  ADR-008 feat_04:    PASS (Goalserve_devig_p_yes_fair not NaN)')
print(f'  Q1 Sharpe bootstrap: PASS ({len(r1)} rows)')
print(f'  Q2 hit rate/edge:    PASS ({len(r2)} rows)')
print(f'  Q3 ML export:        PASS ({len(r3)} sport groups)')
print()
print('Next steps:')
print('  W7+: HC-06 接 ParquetBatchWriter cpp stub 真接 Apache Arrow')
print('  W7+: 老彭 历史 CSV 替换 stub data → 750 万行真实 Parquet')
print('  M2+: 小冯 IngestRaw WAL → 独立 data/ingest_raw/ 分区')